## Creacion de feature pipeline

### By: Carlos Javier Palacios Sanchez

### Date: 06/09/2026

### Description:

Requerimiento
Crear un script en Python llamado feature_pipeline.py que:

Lea los datos originales desde la fuente definida para el proyecto.
Transforme los datos en features adecuados para el modelo de machine learning.
Almacene los features procesados en un archivo.
El script debe poder ejecutarse de forma autónoma.

Puede utilizar como ejemplo: https://github.com/JoseRZapata/air-quality-fti/blob/main/src/pipelines/feature_pipeline/feature-pipeline.py.

Entregables
Script feature_pipeline.py funcional.
Pruebas unitarias.


---

## Solución

El entregable son dos archivos, ubicados según la estructura del proyecto:

| Entregable | Ruta |
|---|---|
| Script | `src/pipelines/feature_pipeline/feature_pipeline.py` |
| Pruebas unitarias | `tests/pipelines/feature_pipeline/test_feature_pipeline.py` |

Este notebook documenta el diseño del pipeline, lo ejecuta de forma autónoma y
verifica sus salidas.

### Criterio de diseño

El script aplica **únicamente transformaciones deterministas y sin ajuste**:
saneamiento de tipos, deduplicación, codificación con vocabulario fijo y atributos
derivados fila a fila.

Las transformaciones que **aprenden parámetros de los datos** —imputación, recorte de
atípicos, escalado, discretización y selección de atributos— permanecen dentro del
`Pipeline` de scikit-learn del *training pipeline*. Aplicarlas aquí, sobre el dataset
completo y antes del `train_test_split`, sería fuga de información (*data leakage*)
hacia el conjunto de prueba: la mediana o la media usadas para imputar y escalar
incorporarían información de las filas de test.

Por esa razón la tabla de features conserva los valores faltantes tal cual.

## 1. Configuración

El notebook vive en `notebooks/9. Creación de Feature Pipeline/`, así que primero se
localiza la raíz del proyecto para que todas las rutas sean independientes del
directorio desde el que se ejecute.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd


def localizar_raiz() -> Path:
    """Sube por el árbol de directorios hasta encontrar el pyproject.toml."""
    actual = Path.cwd().resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pyproject.toml").is_file():
            return candidato
    raise FileNotFoundError("No se encontró la raíz del proyecto")


RAIZ = localizar_raiz()
SCRIPT = RAIZ / "src" / "pipelines" / "feature_pipeline" / "feature_pipeline.py"
PRUEBAS = RAIZ / "tests" / "pipelines" / "feature_pipeline"

ENTRADA = RAIZ / "data" / "01_raw" / "corazon.csv"
INTERMEDIO = RAIZ / "data" / "02_intermediate" / "corazon.parquet"
SALIDA = RAIZ / "data" / "04_feature" / "corazon_features.parquet"
METADATOS = RAIZ / "data" / "04_feature" / "corazon_features_metadata.json"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

print(f"Raíz del proyecto : {RAIZ}")
print(f"Script            : {SCRIPT.relative_to(RAIZ)}  (existe: {SCRIPT.is_file()})")
print(f"Fuente de datos   : {ENTRADA.relative_to(RAIZ)}  (existe: {ENTRADA.is_file()})")

Raíz del proyecto : /mnt/c/Users/KATANA/Heart_project
Script            : src/pipelines/feature_pipeline/feature_pipeline.py  (existe: True)
Fuente de datos   : data/01_raw/corazon.csv  (existe: True)


## 2. Estructura del script

El script no importa nada del propio proyecto (sólo `pandas`, `numpy` y la librería
estándar). Eso le permite ejecutarse directamente sin resolver el paquete `pipelines`
y, a la vez, ser importable por las pruebas.

In [2]:
sys.path.insert(0, str(RAIZ / "src"))

from pipelines.feature_pipeline import feature_pipeline as fp  # noqa: E402

funciones = [
    ("leer_datos_crudos", "lee el CSV crudo como texto y valida el esquema"),
    ("sanear_dataset", "fuerza tipos y valida categorías contra el catálogo del dominio"),
    ("depurar_filas", "elimina duplicados exactos y filas sin variable objetivo"),
    ("codificar_categoricas", "binarias, ordinales y one-hot con vocabulario fijo"),
    ("generar_atributos_clinicos", "6 atributos derivados del conocimiento del dominio"),
    ("construir_features", "orquesta la transformación completa"),
    ("guardar_parquet", "escribe el resultado creando los directorios necesarios"),
    ("guardar_metadatos", "manifiesto JSON de trazabilidad"),
    ("ejecutar_pipeline", "pipeline de extremo a extremo"),
    ("main", "punto de entrada; devuelve 0 si todo va bien y 1 si falla"),
]

print(f"{'función':<28} {'descripción'}")
print("-" * 100)
for nombre, descripcion in funciones:
    print(f"{nombre:<28} {descripcion}")

función                      descripción
----------------------------------------------------------------------------------------------------
leer_datos_crudos            lee el CSV crudo como texto y valida el esquema
sanear_dataset               fuerza tipos y valida categorías contra el catálogo del dominio
depurar_filas                elimina duplicados exactos y filas sin variable objetivo
codificar_categoricas        binarias, ordinales y one-hot con vocabulario fijo
generar_atributos_clinicos   6 atributos derivados del conocimiento del dominio
construir_features           orquesta la transformación completa
guardar_parquet              escribe el resultado creando los directorios necesarios
guardar_metadatos            manifiesto JSON de trazabilidad
ejecutar_pipeline            pipeline de extremo a extremo
main                         punto de entrada; devuelve 0 si todo va bien y 1 si falla


### Configuración del dominio

El catálogo de categorías válidas es el que se definió en el notebook `4-feat_eng`:
cualquier valor fuera de él se considera basura y se convierte en faltante.

In [3]:
print("Columnas numéricas:")
print(" ", fp.COLS_NUMERICAS)
print("\nCatálogo de categorías válidas:")
for columna, validas in fp.CATEGORIAS_VALIDAS.items():
    print(f"  {columna:<12} -> {validas}")
print(f"\nVariable objetivo: {fp.OBJETIVO}")

Columnas numéricas:
  ['age', 'rest_bp', 'chol', 'max_hr', 'old_peak', 'ca', 'fbs']

Catálogo de categorías válidas:
  sex          -> ['female', 'male']
  chest_pain   -> ['asymptomatic', 'nonanginal', 'nontypical', 'typical']
  rest_ecg     -> ['left ventricular hypertrophy', 'normal', 'st-t wave abnormality']
  exang        -> ['0', '1']
  slope        -> ['1', '2', '3']
  thal         -> ['fixed', 'normal', 'reversable']
  disease      -> ['0', '1']

Variable objetivo: disease


## 3. Ejecución autónoma

Se invoca el script como proceso independiente —igual que lo haría un `cron`, un
`Makefile` o un job de CI— para comprobar el requisito de que **debe poder ejecutarse
de forma autónoma**. No recibe argumentos: resuelve por sí mismo la raíz del proyecto
y las rutas de entrada y salida.

In [4]:
ejecucion = subprocess.run(
    [sys.executable, str(SCRIPT)],
    capture_output=True,
    text=True,
    check=False,
)

# El registro (logging) se emite por stderr
print(ejecucion.stderr or ejecucion.stdout)
print(f"Código de salida: {ejecucion.returncode}  ->  {'OK' if ejecucion.returncode == 0 else 'FALLO'}")

15:46:40 | INFO     | === Feature pipeline: inicio ===
15:46:40 | INFO     | Datos crudos leídos desde /mnt/c/Users/KATANA/Heart_project/data/01_raw/corazon.csv -> 3030 filas x 14 columnas
15:46:40 | INFO     | Saneamiento de tipos: 33 valores inválidos convertidos a NaN
15:46:40 | INFO     | Duplicados eliminados: 2462 filas (3030 -> 568)
15:46:40 | INFO     | Filas sin etiqueta eliminadas: 88 -> dataset final 480 filas
15:46:40 | INFO     | Features construidos: 27 columnas a partir de 14 columnas originales
15:46:40 | INFO     | Archivo escrito: /mnt/c/Users/KATANA/Heart_project/data/02_intermediate/corazon.parquet (480 filas x 14 columnas)
15:46:40 | INFO     | Archivo escrito: /mnt/c/Users/KATANA/Heart_project/data/04_feature/corazon_features.parquet (480 filas x 27 columnas)
15:46:41 | INFO     | Metadatos escritos: /mnt/c/Users/KATANA/Heart_project/data/04_feature/corazon_features_metadata.json
15:46:41 | INFO     | === Feature pipeline: fin (480 filas listas) ===

Código de sal

## 4. Trazabilidad del saneamiento

Se reconstruye paso a paso el efecto de cada etapa sobre el número de filas, para
documentar cuántos registros se pierden y por qué.

In [5]:
crudo = fp.leer_datos_crudos(ENTRADA)
saneado = fp.sanear_dataset(crudo)
depurado = fp.depurar_filas(saneado)

basura = int(saneado.isna().sum().sum() - crudo.isna().sum().sum())

resumen = pd.DataFrame(
    [
        ("Archivo original", len(crudo), "—"),
        ("Tras sanear tipos", len(saneado), f"{basura} valores basura -> NaN"),
        ("Tras depurar filas", len(depurado), "duplicados y filas sin etiqueta"),
    ],
    columns=["etapa", "filas", "observación"],
)
resumen["% conservado"] = (resumen["filas"] / len(crudo) * 100).round(1)
resumen

,etapa,filas,observación,% conservado
0,Archivo original,3030,—,100.0
1,Tras sanear tipos,3030,33 valores basura -> NaN,100.0
2,Tras depurar filas,480,duplicados y filas sin etiqueta,15.8


In [6]:
print("Distribución de la clase objetivo tras la depuración:")
conteo = depurado[fp.OBJETIVO].value_counts().rename({0: "sano (0)", 1: "enfermo (1)"})
print(conteo.to_string())
print(f"\nBalance: {depurado[fp.OBJETIVO].mean():.1%} de casos positivos")

Distribución de la clase objetivo tras la depuración:
disease
sano (0)       250
enfermo (1)    230

Balance: 47.9% de casos positivos


## 5. Verificación de la tabla de features

Se carga el parquet que dejó el script y se comprueba que cumple lo pedido: un
archivo con los features procesados, enteramente numérico y listo para el modelo.

In [7]:
features = pd.read_parquet(SALIDA)

print(f"Archivo   : {SALIDA.relative_to(RAIZ)}")
print(f"Dimensión : {features.shape[0]} filas x {features.shape[1]} columnas")
print(f"Todas las columnas son numéricas: {all(pd.api.types.is_numeric_dtype(features[c]) for c in features.columns)}")
features.head()

Archivo   : data/04_feature/corazon_features.parquet
Dimensión : 480 filas x 27 columnas
Todas las columnas son numéricas: True


,age,rest_bp,chol,max_hr,old_peak,ca,fbs,sex,exang,slope,chest_pain_asymptomatic,chest_pain_nonanginal,chest_pain_nontypical,chest_pain_typical,rest_ecg_left_ventricular_hypertrophy,rest_ecg_normal,rest_ecg_st_t_wave_abnormality,thal_fixed,thal_normal,thal_reversable,fc_maxima_teorica,reserva_cardiaca,pct_fc_alcanzada,ratio_chol_edad,presion_x_chol,indice_riesgo_st,disease
0,63.0,145.0,233.0,150.0,2.3,0.0,1.0,1.0,0.0,2.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,157.0,-7.0,0.955414,3.698413,33.785,4.6,0
1,67.0,160.0,286.0,108.0,1.5,3.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,153.0,-45.0,0.705882,4.268657,45.760,1.5,1
2,67.0,120.0,229.0,129.0,2.6,2.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,153.0,-24.0,0.843137,3.417910,27.480,2.6,1
3,37.0,130.0,250.0,187.0,3.5,0.0,0.0,1.0,0.0,2.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,183.0,4.0,1.021858,6.756757,32.500,7.0,0
4,41.0,130.0,204.0,172.0,1.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,179.0,-7.0,0.960894,4.975610,26.520,0.0,0


In [8]:
grupos = {
    "Numéricas originales": fp.COLS_PASO_DIRECTO,
    "Binarias / ordinales": list(fp.MAPA_BINARIAS) + list(fp.ORDEN_ORDINALES),
    "One-hot nominales": [c for c in features.columns if c.startswith(tuple(fp.COLS_NOMINALES))],
    "Atributos derivados": [
        "fc_maxima_teorica",
        "reserva_cardiaca",
        "pct_fc_alcanzada",
        "ratio_chol_edad",
        "presion_x_chol",
        "indice_riesgo_st",
    ],
    "Objetivo": [fp.OBJETIVO],
}

for grupo, columnas in grupos.items():
    print(f"{grupo} ({len(columnas)}):")
    print(f"  {columnas}\n")

print(f"TOTAL: {features.shape[1]} columnas")

Numéricas originales (7):
  ['age', 'rest_bp', 'chol', 'max_hr', 'old_peak', 'ca', 'fbs']

Binarias / ordinales (3):
  ['sex', 'exang', 'slope']

One-hot nominales (10):
  ['chest_pain_asymptomatic', 'chest_pain_nonanginal', 'chest_pain_nontypical', 'chest_pain_typical', 'rest_ecg_left_ventricular_hypertrophy', 'rest_ecg_normal', 'rest_ecg_st_t_wave_abnormality', 'thal_fixed', 'thal_normal', 'thal_reversable']

Atributos derivados (6):
  ['fc_maxima_teorica', 'reserva_cardiaca', 'pct_fc_alcanzada', 'ratio_chol_edad', 'presion_x_chol', 'indice_riesgo_st']

Objetivo (1):
  ['disease']

TOTAL: 27 columnas


### Atributos derivados

Los seis atributos clínicos provienen del notebook `4-feat_eng` y codifican
conocimiento del dominio que el modelo no puede inferir por sí solo de las columnas
crudas:

| Atributo | Fórmula | Interpretación |
|---|---|---|
| `fc_maxima_teorica` | `220 - age` | frecuencia cardíaca máxima esperada por edad |
| `reserva_cardiaca` | `max_hr - fc_maxima_teorica` | cuánto se aleja el paciente de su máximo teórico |
| `pct_fc_alcanzada` | `max_hr / fc_maxima_teorica` | proporción del máximo teórico alcanzada |
| `ratio_chol_edad` | `chol / age` | colesterol relativo a la edad |
| `presion_x_chol` | `rest_bp * chol / 1000` | interacción presión-colesterol |
| `indice_riesgo_st` | `old_peak * (slope - 1)` | depresión del ST ponderada por la pendiente |

In [9]:
derivados = [
    "fc_maxima_teorica",
    "reserva_cardiaca",
    "pct_fc_alcanzada",
    "ratio_chol_edad",
    "presion_x_chol",
    "indice_riesgo_st",
]
features[derivados].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
fc_maxima_teorica,463.0,165.428,8.901,143.000,159.000,164.000,172.000,191.000
reserva_cardiaca,378.0,-15.630,20.599,-82.000,-27.000,-12.000,0.000,29.000
pct_fc_alcanzada,378.0,0.905,0.125,0.464,0.840,0.929,1.000,1.175
ratio_chol_edad,391.0,4.620,1.106,2.099,3.836,4.451,5.298,8.418
presion_x_chol,404.0,32.695,8.635,15.228,26.475,31.915,37.500,64.860
indice_riesgo_st,406.0,1.105,1.888,0.000,0.000,0.000,1.750,12.400


### Valores faltantes

Los faltantes **se conservan a propósito**: imputarlos aquí, sobre el dataset
completo, contaminaría el conjunto de prueba. La imputación se hace dentro del
`Pipeline` de entrenamiento, ajustada sólo con los datos de *train*.

In [10]:
faltantes = (
    features.isna().sum().rename("faltantes").to_frame().assign(
        porcentaje=lambda d: (d["faltantes"] / len(features) * 100).round(1)
    )
)
faltantes[faltantes["faltantes"] > 0].sort_values("faltantes", ascending=False)

,faltantes,porcentaje
rest_ecg_st_t_wave_abnormality,116,24.2
rest_ecg_normal,116,24.2
rest_ecg_left_ventricular_hypertrophy,116,24.2
reserva_cardiaca,102,21.2
pct_fc_alcanzada,102,21.2
max_hr,90,18.8
ratio_chol_edad,89,18.5
fbs,82,17.1
presion_x_chol,76,15.8
indice_riesgo_st,74,15.4


## 6. Manifiesto de trazabilidad

Además del parquet, el script escribe un JSON con la marca de tiempo, la fuente y el
esquema generado. Sirve para saber, meses después, con qué datos y qué columnas se
entrenó un modelo.

In [11]:
manifiesto = json.loads(METADATOS.read_text(encoding="utf-8"))

print(f"Generado en : {manifiesto['generado_en']}")
print(f"Fuente      : {Path(manifiesto['fuente']).relative_to(RAIZ)}")
print(f"Dimensión   : {manifiesto['n_filas']} x {manifiesto['n_columnas']}")
print(f"Objetivo    : {manifiesto['objetivo']}")
print(f"Columnas con faltantes: {len(manifiesto['faltantes_por_columna'])}")

Generado en : 2026-09-06T20:46:40+00:00
Fuente      : data/01_raw/corazon.csv
Dimensión   : 480 x 27
Objetivo    : disease
Columnas con faltantes: 26


## 7. Pruebas unitarias

Las 26 pruebas cubren: utilidades, lectura y validación de esquema, saneamiento,
depuración de filas, codificación, atributos derivados, determinismo, escritura de
parquet y metadatos, y dos pruebas de extremo a extremo sobre `main()`.

In [12]:
pruebas = subprocess.run(
    [sys.executable, "-m", "pytest", str(PRUEBAS), "-v", "--no-header"],
    cwd=RAIZ,
    capture_output=True,
    text=True,
    check=False,
)

print(pruebas.stdout[-4000:])
print(f"Código de salida: {pruebas.returncode}  ->  {'TODAS PASAN' if pruebas.returncode == 0 else 'HAY FALLOS'}")

============================= test session starts ==============================
collecting ... collected 26 items

tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_normalizar_nombre_convierte_a_sufijo_valido PASSED [  3%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_localizar_raiz_encuentra_pyproject PASSED [  7%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_leer_datos_crudos_lee_todo_como_texto PASSED [ 11%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_leer_datos_crudos_falla_si_no_existe PASSED [ 15%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_leer_datos_crudos_falla_si_faltan_columnas PASSED [ 19%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_sanear_convierte_basura_numerica_en_nan PASSED [ 23%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_sanear_normaliza_categorias PASSED [ 26%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_sanear_de

## 8. Conclusiones

El requerimiento queda cubierto en sus cuatro puntos:

1. **Lee los datos originales desde la fuente definida para el proyecto** —
   `data/01_raw/corazon.csv`, leído como texto para no perder los valores no
   interpretables antes de poder detectarlos.
2. **Transforma los datos en features adecuados para el modelo** — 27 columnas
   numéricas: las originales saneadas, las categóricas codificadas con vocabulario
   fijo y los 6 atributos clínicos derivados.
3. **Almacena los features procesados en un archivo** —
   `data/04_feature/corazon_features.parquet`, más el parquet intermedio saneado y un
   manifiesto JSON de trazabilidad.
4. **Se ejecuta de forma autónoma** — sin argumentos, resolviendo por sí mismo las
   rutas del proyecto y devolviendo un código de salida apto para automatización.

La decisión de diseño que vale la pena subrayar es la separación entre
transformaciones deterministas (aquí) y transformaciones que aprenden de los datos
(en el *training pipeline*). Es lo que permite que este pipeline se pueda ejecutar
sobre datos nuevos en producción, con las mismas reglas, sin necesidad de reajustar
nada y sin contaminar la evaluación del modelo.